In [0]:
%python
dbutils.widgets.text("date_partition", "2023-10-26", "Date Partition")
date_to_process = dbutils.widgets.get("date_partition")
print(f"Processing data for: {date_to_process}")

**EDA**

In [0]:
%python
#eda --> load into a dataframe 

df = spark.read.csv("/Volumes/dev/academy/data/Task W3_U3 rental-price-indexes-september-2023.csv", header=True, inferSchema=True)
print(f"Total rows: {df.count()}, Total columns: {len(df.columns)}")

display(df.limit(5))

In [0]:
%python
# Print schema (column names and data types)
df.printSchema()

In [0]:
%python
# Summary statistics for numeric columns
display(df.describe())


In [0]:
%python
from pyspark.sql.functions import sum, when, col

# Count null values per column
null_counts_df = df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
])
display(null_counts_df)


In [0]:
%python
total_rows = df.count()
distinct_rows = df.dropDuplicates().count()
duplicate_count = total_rows - distinct_rows
print(f"Duplicate rows: {duplicate_count}")


Since the date column has a value that doesnt match the standar we should normalize it to work with it

In [0]:
%python
from pyspark.sql.functions import col, to_date, concat_ws, lit, split,lpad
# Assigning to a new DataFrame is a safe practice in notebooks
df_with_date = df.withColumn(
    "DATE_REF",
    to_date(
        # Concatenate parts into a standard 'YYYY-MM-01' string
        concat_ws("-",
            split(col("TIME_REF").cast("string"), "\\.")[0], # The part before the '.' is the year
            lpad(split(col("TIME_REF").cast("string"), "\\.")[1], 2, '0'), # The part after the '.' is the month
            lit("01")                                      # A literal '01' for the day
        ),
        "yyyy-MM-dd" # The format of the string we just built
    )
).orderBy(col("DATE_REF").asc())

display(df_with_date)
df_with_date.printSchema()

**Basic Level Filters and Select Querys**

In [0]:
%python
#How many records for Auckland are there since January 2020?
from pyspark.sql.functions import col

df_with_date.where(
    (col("Series_title_1") == 'Auckland') & 
    (col("DATE_REF") >= '2020-01-01')
).count()

In [0]:
%python
# In which months did Auckland and Wellington exceed an index of 1200?
from pyspark.sql.functions import col

df_with_date.where(
    (col("Series_title_1").isin(['Auckland', 'Wellington'])) & 
    (col("DATA_VAL") > 1200)
).select("Series_title_1", "DATE_REF").distinct().orderBy("DATE_REF").display(truncate=False)

In [0]:
%python
from pyspark.sql.functions import col

#What are the 10 highest index values recorded in 2023?

#ask whats the best approach handling dates, using year() or other functions
df_with_date.where(col("DATE_REF").startswith("2023")).select("SER_REF","DATA_VAL").orderBy(col("DATA_VAL").desc()).limit(10).display(truncate=False)

**Intermediate Level Agregations**

In [0]:
%python
from pyspark.sql.functions import col,avg
#Which region has the highest average index historically?
#considering that the series_title_1 column contains the region name and also has a national value that cant be taken into proccesing 

df_with_date.where(col("Series_title_1") != 'National').groupBy("Series_title_1").agg(avg("DATA_VAL").alias("Average")).orderBy(col("Average").desc()).limit(1).display(truncate=False)

In [0]:
%python
from pyspark.sql.functions import col, year,max, min
#In which year was there the greatest variation between the maximum and minimum index?

df_with_date.groupBy(year("DATE_REF")).agg(max("DATA_VAL").alias("max"),min("DATA_VAL").alias("min")).withColumn("diff",col("max")-col("min")).orderBy(col("diff").desc()).limit(1).display(truncate=False)

In [0]:
%python
from pyspark.sql.functions import col

#How many records with a "FINAL" status does each region have?
#should probably filter out national?
df_with_date.where(col("STATUS") == "FINAL").groupBy("Series_title_1").count().display(truncate=False)

In [0]:
%python
from pyspark.sql.functions import col, round
#How did the national annual average index evolve year over year?
# only consider series title 1 as national 
df_with_date.where(col("Series_title_1") == 'National').groupBy(year("DATE_REF")).agg(round(avg("DATA_VAL"), 2).alias("Average")).orderBy(col("Average").desc()).display(truncate=False)
       


**Advanced Level Analytical and Window functions**

In [0]:
%python
from pyspark.sql.functions import col, month, rank
from pyspark.sql.window import Window

# Which region had the highest index each month during 2023?

# First get the 2023 subset
df_2023 = df_with_date.where(col("DATE_REF").startswith("2023"))

windowSpec = Window.partitionBy(month("DATE_REF")).orderBy(col("DATA_VAL").desc())
df_2023 = df_2023.withColumn("rank", rank().over(windowSpec)).filter(col("rank") == 1)
df_2023.select("SER_REF","Series_title_1","DATA_VAL","DATE_REF").display()

In [0]:
%python
from pyspark.sql.functions import col, month, lag
#What was the monthly change in Auckland's index over time?

#So here i have to compare the DATA_VAL between each row doing something like previous_DATA_VAL - DATA VAL

WindowSpec = Window.partitionBy("Series_title_1").orderBy(col("DATE_REF"))

df_auckland = df_with_date.withColumn("previous_DATA_VAL", lag("DATA_VAL", 1).over(WindowSpec)).withColumn("change", col("DATA_VAL") - col("previous_DATA_VAL")).filter(col("Series_title_1") == "Auckland") #Here there should be another filter but i cant find wich column use (because there are multiple rows for one month)
df_auckland.select("SER_REF","Series_title_1","DATA_VAL","DATE_REF","change").display()
       


In [0]:
%python
from pyspark.sql.functions import col, first, last
#What was the first and last index value recorded for each region?

#GroupBy solves the issue

df_with_date.groupBy("Series_title_1").agg(first("DATA_VAL").alias("first"),last("DATA_VAL").alias("last")).display()

In [0]:
%python
from pyspark.sql.functions import col, countDistinct

#What is the cumulative percentage growth for Wellington from 2006 to 2023?

df_wellington = df_with_date.filter((col("Series_title_1") == "Wellington") & (year("DATE_REF").between(2006, 2023)))
window_spec = Window.partitionBy("Series_title_1").orderBy("DATE_REF").rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

df_with_start_value = df_wellington.withColumn(
    "start_value", 
    first("DATA_VAL").over(window_spec)
    ).withColumn(
    "cumulative_growth_pct",
    round(
        ((col("DATA_VAL") - col("start_value")) / col("start_value")) * 100, 
        2  # Round to 2 decimal places for readability
    )).select("DATE_REF", "DATA_VAL", "start_value", "cumulative_growth_pct").display()



In [0]:
%python 
from pyspark.sql.functions import col, lead
# In which months did Auckland register local maximum peaks (values higher than the previous and following month)?

df_auckland = df_with_date.filter(col("Series_title_1") == "Auckland")
windowSpec = Window.partitionBy("Series_title_1").orderBy("DATE_REF")
# Instead of using the RowsBetween i use the lead and lag functions provided 
df_auckland = df_auckland.withColumn(
    "previous_month_val", lag("DATA_VAL", 1).over(windowSpec)
).withColumn(
    "next_month_val", lead("DATA_VAL", 1).over(windowSpec)
).filter(
    (col("DATA_VAL") > col("previous_month_val")) & 
    (col("DATA_VAL") > col("next_month_val"))
).select("DATE_REF", "previous_month_val", "DATA_VAL", "next_month_val").display()



In [0]:
%python
# Get the months where a certain region surpassed a minimum value index

region = dbutils.widgets.get("Region") 
min_value = dbutils.widgets.get("Min_Index_Val")

df_with_date.where((col("Series_title_1") == region )& (col("DATA_VAL") > min_value)).select("Series_title_1","DATE_REF", "DATA_VAL").display()